# **Aprendizaje por refuerzos** - FrozenLake y LunarLander (Gymnasium)

## Tarea: Implementar Agentes Q-Learning y DQN

### Objetivos:
1. Implementar el algoritmo Q-Learning
2. Implementar el algoritmo DQN
3. Entrenar y evaluar ambos agentes
4. Comparar el rendimiento de ambos enfoques


In [ ]:
# Instalar paquetes requeridos
import os
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE"

%pip install swig matplotlib gymnasium torch pygame


In [1]:
# Importar las bibliotecas
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
import pygame
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from collections import deque, namedtuple
import random


La siguiente celda permite ejecutar un juego de Frozen Lake *determinista* para jugar con el teclado.

Utilize las teclas de dirección (flechas) o asdw para comandar al agente.


In [12]:
def jugar_frozen_lake(env):
    env.reset()
    
    print("Controles:")
    print("W - Arriba")
    print("S - Abajo") 
    print("A - Izquierda")
    print("D - Derecha")
    print("Q - Salir")
    print("Presione cualquier tecla para empezar...")
    
    pygame.init()
    pygame.display.set_caption("FrozenLake - Juego Interactivo")
    
    clock = pygame.time.Clock()
    ejecutando = True
    
    while ejecutando:
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                ejecutando = False
            elif event.type == pygame.KEYDOWN:
                if event.key == pygame.K_q or event.key == pygame.K_ESCAPE:
                    ejecutando = False
                elif event.key == pygame.K_w or event.key == pygame.K_UP:
                    accion = 3  # Arriba
                elif event.key == pygame.K_s or event.key == pygame.K_DOWN:
                    accion = 1  # Abajo
                elif event.key == pygame.K_a or event.key == pygame.K_LEFT:
                    accion = 0  # Izquierda
                elif event.key == pygame.K_d or event.key == pygame.K_RIGHT:
                    accion = 2  # Derecha
                else:
                    continue
                
                observacion, recompensa, terminado, truncado, info = env.step(accion)
                print(f"Acción: {accion}, Recompensa: {recompensa}, Terminado: {terminado}")
                
                if terminado or truncado:
                    print(f"¡Episodio terminado! Recompensa final: {recompensa}")
                    pygame.time.wait(500)
                    env.reset()
        
        clock.tick(60)
    
    pygame.quit()
    env.close()

env = gym.make('FrozenLake-v1', render_mode='human', is_slippery=False)
# Descomente la línea de abajo para jugar interactivamente
# jugar_frozen_lake(env)


La siguiente celda permite jugar al juego no determinista.

In [ ]:
env = gym.make('FrozenLake-v1', render_mode='human', is_slippery=True)
# Descomente la línea de abajo para jugar interactivamente
jugar_frozen_lake(env)

La siguiente clase define la interfaz de los agentes que utilizaremos para jugar al Frozen Lake.


In [2]:
from abc import ABC, abstractmethod

class Agente(ABC):
    
    @abstractmethod
    def elegir_accion(self, estado):
        """Elige una acción dada una observación."""
        pass
    
    @abstractmethod
    def aprender(self, estado, accion, recompensa, siguiente_estado, terminado):
        """Aprende de la experiencia."""
        pass

class AgenteAleatorio(Agente):
    """Agente aleatorio que elige acciones al azar."""
    
    def __init__(self, espacio_acciones):
        # Se guarda el espacio de acciones para poder elegir acciones al azar
        self.espacio_acciones = espacio_acciones
    
    def elegir_accion(self, estado):
        return self.espacio_acciones.sample()
    
    def aprender(self, estado, accion, recompensa, siguiente_estado, terminado):
        pass  # El agente aleatorio no aprende

# Probar el AgenteAleatorio
env = gym.make('FrozenLake-v1')
agente_aleatorio = AgenteAleatorio(env.action_space)
estado, _ = env.reset()
accion = agente_aleatorio.elegir_accion(estado)
print(f"✓ AgenteAleatorio creado y probado. Acción: {accion}")
env.close()


✓ AgenteAleatorio creado y probado. Acción: 1


La siguiente celda define una función para evaluar el desempeño de un agente dado.

In [3]:
# Función de Evaluación de Agentes
def evaluar_agente(agente, env, num_episodios=1000):
    """
    Evalúa el rendimiento de un agente a lo largo de múltiples episodios.
    
    Args:
        agente: El agente a evaluar
        env: El entorno
        num_episodios: Número de episodios a ejecutar
    
    Returns:
        dict: Resultados de la evaluación
    """
    recompensas_totales = []
    victorias = 0
    
    for episodio in range(num_episodios):
        estado, _ = env.reset()
        recompensa_total = 0
        
        while True:
            accion = agente.elegir_accion(estado)
            estado, recompensa, terminado, truncado, _ = env.step(accion)
            recompensa_total += recompensa
            
            if terminado or truncado:
                break
        
        recompensas_totales.append(recompensa_total)
        if recompensa_total > 0:
            victorias += 1
    
    return {
        'recompensas_totales': recompensas_totales,
        'victorias': victorias,
        'tasa_victorias': victorias / num_episodios,
        'recompensa_promedio': np.mean(recompensas_totales),
        'desv_estandar': np.std(recompensas_totales)
    }

def imprimir_resultados_evaluacion(resultados, nombre_agente):
    """Imprime los resultados de evaluación de forma formateada."""
    print(f"\n{nombre_agente} - Resultados de Evaluación:")
    print(f"Tasa de Victorias: {resultados['tasa_victorias']:.1%}")
    print(f"Recompensa Promedio: {resultados['recompensa_promedio']:.3f}")
    print(f"Desviación Estándar: {resultados['desv_estandar']:.3f}")
    print(f"Total de Victorias: {resultados['victorias']}")

# Probar función de evaluación
env = gym.make('FrozenLake-v1')
agente_aleatorio = AgenteAleatorio(env.action_space)
resultados = evaluar_agente(agente_aleatorio, env, num_episodios=100)
imprimir_resultados_evaluacion(resultados, "Agente Aleatorio")
env.close()



Agente Aleatorio - Resultados de Evaluación:
Tasa de Victorias: 4.0%
Recompensa Promedio: 0.040
Desviación Estándar: 0.196
Total de Victorias: 4


La siguiente celda define una función para entrenar un agente.

In [4]:
# Función de Entrenamiento de Agentes
def entrenar_agente(agente, env, num_episodios=1000, max_pasos=100, verbose=True):
    """
    Entrena un agente en el entorno.
    
    Args:
        agente: El agente a entrenar
        env: El entorno
        num_episodios: Número de episodios de entrenamiento
        max_pasos: Máximo de pasos por episodio
        verbose: Si imprimir el progreso
    
    Returns:
        list: Recompensas de episodios
    """
    recompensas_episodios = []
    longitudes_episodios = []
    num_episodios_10 = int(num_episodios / 10)
    
    for episodio in range(num_episodios):
        estado, _ = env.reset()
        recompensa_total = 0
        pasos = 0
        
        for paso in range(max_pasos):
            accion = agente.elegir_accion(estado)
            siguiente_estado, recompensa, terminado, truncado, _ = env.step(accion)
            
            agente.aprender(estado, accion, recompensa, siguiente_estado, terminado or truncado)
            
            estado = siguiente_estado
            recompensa_total += recompensa
            pasos += 1
            
            if terminado or truncado:
                break
        
        recompensas_episodios.append(recompensa_total)
        longitudes_episodios.append(pasos)
        
        if verbose and (episodio + 1) % num_episodios_10 == 0:
            recompensa_promedio = np.mean(recompensas_episodios[-num_episodios_10:])
            longitud_promedio = np.mean(longitudes_episodios[-num_episodios_10:])
            print(f"Episodio {episodio + 1}: Recompensa Promedio = {recompensa_promedio:.3f}, Longitud Promedio = {longitud_promedio:.1f}")
    
    return recompensas_episodios, longitudes_episodios

print("✓ Funciones de entrenamiento definidas")


✓ Funciones de entrenamiento definidas


In [7]:
# Ejecución del Agente Aleatorio
env = gym.make('FrozenLake-v1')
agente_aleatorio = AgenteAleatorio(env.action_space)
entrenar_agente(agente_aleatorio, env)
resultados = evaluar_agente(agente_aleatorio, env, num_episodios=1000)
imprimir_resultados_evaluacion(resultados, "Agente Aleatorio")
env.close()

Episodio 100: Recompensa Promedio = 0.000, Longitud Promedio = 7.7
Episodio 200: Recompensa Promedio = 0.020, Longitud Promedio = 7.1
Episodio 300: Recompensa Promedio = 0.030, Longitud Promedio = 7.7
Episodio 400: Recompensa Promedio = 0.010, Longitud Promedio = 8.2
Episodio 500: Recompensa Promedio = 0.020, Longitud Promedio = 7.9
Episodio 600: Recompensa Promedio = 0.020, Longitud Promedio = 7.3
Episodio 700: Recompensa Promedio = 0.010, Longitud Promedio = 7.2
Episodio 800: Recompensa Promedio = 0.020, Longitud Promedio = 7.9
Episodio 900: Recompensa Promedio = 0.020, Longitud Promedio = 7.3
Episodio 1000: Recompensa Promedio = 0.020, Longitud Promedio = 6.4

Agente Aleatorio - Resultados de Evaluación:
Tasa de Victorias: 1.7%
Recompensa Promedio: 0.017
Desviación Estándar: 0.129
Total de Victorias: 17


La siguiente celda define el agente de Q-Learning a implementar.

In [24]:
# TODO: Implementar Agente Q-Learning
class AgenteQLearning(Agente):
    """Agente que usa el algoritmo Q-Learning."""
    
    def __init__(self):
        pass
    
    def elegir_accion(self, estado):
        """Elige una acción usando política epsilon-greedy."""
        pass
    
    def aprender(self, estado, accion, recompensa, siguiente_estado, terminado):
        """Actualiza la tabla Q usando la ecuación de Bellman."""
        pass


Código para entrenar y evaluar el agente de Q-Learning implementado.

In [ ]:
# Ejecución del Agente QLearning Determinista
env = gym.make('FrozenLake-v1')
agente_qlearning = AgenteQLearning(env.action_space)
entrenar_agente(agente_qlearning, env)
resultados = evaluar_agente(agente_qlearning, env, num_episodios=100)
imprimir_resultados_evaluacion(resultados, "Agente QLearning")
env.close()

La siguiente celda define el agente DQN a implementar. 

In [ ]:
## Version que implementa memoria de experiencia y replay buffer

class DQN(nn.Module):
    """Clase auxiliar que implementa una Red Q Profunda con una capa oculta."""
    
    def __init__(self, tamano_entrada, tamano_oculto, tamano_salida):
        super(DQN, self).__init__()
        # Conexiones capa de entrada a oculta
        self.fc1 = nn.Linear(tamano_entrada, tamano_oculto)
        # Conexiones capa oculta a salida
        self.fc2 = nn.Linear(tamano_oculto, tamano_salida)
    
    def forward(self, x):
        # Se aplica ReLU a la combinacion lineal que llega a la capa oculta
        x = F.relu(self.fc1(x))  # JUSTIFICAR PORQUE USAMOS RELU
        # Se retorna la combinacion lineal que llega a la capa de salida
        # (no se aplica activación en la capa de salida)
        return self.fc2(x)

class AgenteDQN(Agente):
    """Agente de Red Q Profunda."""
    TAMANO_ENTRADA = 64     # Tamaño del estado (FrozenLake-v1 con mapa 4x4)
    TAMANO_OCULTO = 128     # Tamaño de la capa oculta
    TAMANO_SALIDA = 4       # Número de acciones posibles
    MEMORIA_MAX = 10000     # Tamaño máximo de la memoria de experiencias
    BATCH_SIZE = 64         # Tamaño del mini-batch para el replay buffer
    CONTADOR = 1000         # Cada cuántos pasos se actualiza la red objetivo
    LR = 0.00025            # Tasa de aprendizaje
    EPSILON_INICIAL = 1.0
    EPSILON_FINAL = 0.03
    EPSILON_DECAY = 0.999
    GAMMA = 0.95            # Factor de descuento

    def __init__(self):
        self.model = DQN(self.TAMANO_ENTRADA, self.TAMANO_OCULTO, self.TAMANO_SALIDA)
        # Creamos una copia de la red en su estado actual (red objetivo) para calcular el objetivo (target).
        # Esto es útil para estabilizar el entrenamiento. PROBAR QUE FUNCIONA MEJOR
        self.target_model = DQN(self.TAMANO_ENTRADA, self.TAMANO_OCULTO, self.TAMANO_SALIDA)
        self.target_model.load_state_dict(self.model.state_dict())
        # Seteamos la red objetivo en modo evaluación, pues solo se usa para inferencia
        self.target_model.eval()  # JUSTIFICAR POR QUE ES MAS EFICIENTE
        self.counter = self.CONTADOR
        self.epsilon = self.EPSILON_INICIAL
        # Creamos una memoria de experiencias
        self.memoria = deque(maxlen=self.MEMORIA_MAX)
        # self.criterion = nn.MSELoss()  # JUSTIFICAR PORQUE USAMOS MSELoss
        self.criterion = nn.SmoothL1Loss()  # OTRA OPCIÓN, más suave
        self.optimizer = optim.Adam(self.model.parameters(), lr=self.LR)  # JUSTIFICAR POR QUE USAMOS Adam

    def actualizar_red_objetivo(self):
        """Cada CONTADOR pasos actualiza los pesos de la red objetivo con los pesos de la red principal."""
        self.counter -= 1
        if self.counter == 0:
            self.target_model.load_state_dict(self.model.state_dict())
            self.counter = self.CONTADOR

    def guardar_experiencia(self, estado, accion, recompensa, siguiente_estado, terminado):
        self.memoria.append((estado, accion, recompensa, siguiente_estado, terminado))
    
    def elegir_accion(self, estado):
        """Elige acción usando política epsilon-greedy."""
        # Tomamos una acción aleatoria con probabilidad epsilon (exploración)
        if random.random() < self.epsilon:
            accion = np.random.randint(self.TAMANO_SALIDA)
        #
        # Tomamos la acción con mayor Q-valor con probabilidad 1 - epsilon (explotación)
        else:
            # El estado es un integer del 0 al 15, se convierte a vector mediante one-hot encoding
            vector_estado = np.identity(self.TAMANO_ENTRADA)[estado]
            # El vector se convierte a tensor de PyTorch para poder ser procesado por la red.
            # Debe ser de tipo FloatTensor (para que pueda se operar con el) y tener una dimensión 
            # extra para el batch (El batch es el número de tensores de entrada que la red neuronal
            # procesa en paralelo durante una sola pasada hacia adelante y hacia atrás)
            tensor_estado = torch.FloatTensor(vector_estado).unsqueeze(0)
            # Indicamos a PyTorch que no construya el grafo computacional que usaría para calcular
            # gradientes, ya que en este paso no vamos a hacer backpropagation y es innecesario
            with torch.no_grad():
                # Pasamos el estado por la red para obtener los Q-valores
                q_valores = self.model(tensor_estado)
                # Elige la acción con el mayor Q-valor
                accion = torch.argmax(q_valores).item()
        # Actualizamos epsilon luego de elegir una acción
        self.epsilon = max(self.EPSILON_FINAL, self.epsilon * self.EPSILON_DECAY)
        # Retornamos la acción elegida
        return accion
    
    def aprender(self, estado, accion, recompensa, siguiente_estado, terminado):
        """Actualiza la red neuronal usando la experiencia (s, a, r, s')."""
        # Primero que nada, guardamos la experiencia en la memoria
        self.guardar_experiencia(estado, accion, recompensa, siguiente_estado, terminado)
        # Actualizamos, si corresponde, la red objetivo
        self.actualizar_red_objetivo()
        # Hasta no tener un batch completo, no entrenamos
        if len(self.memoria) < self.BATCH_SIZE:
            return
        # 0 - Limpiamos los gradientes acumulados antes de la actualización
        self.optimizer.zero_grad()
        # 1 - Tomamos un mini-batch aleatorio de la memoria y generamos tensores
        batch = random.sample(self.memoria, self.BATCH_SIZE)
        estados, acciones, recompensas, siguientes_estados, terminados = zip(*batch)
        # Convertimos a tensores
        estados = torch.FloatTensor(np.identity(self.TAMANO_ENTRADA)[list(estados)])
        acciones = torch.LongTensor(acciones).unsqueeze(1)
        recompensas = torch.FloatTensor(recompensas).unsqueeze(1)
        siguientes_estados = torch.FloatTensor(np.identity(self.TAMANO_ENTRADA)[list(siguientes_estados)])
        terminados = torch.FloatTensor(terminados).unsqueeze(1)
        # 2 - Calculamos los Q-valores para estados actuales
        q_valores = self.model(estados)
        # Obtenemos los Q-valores correspondientes a la acciones tomadas en el batch
        q_valor_accion = q_valores.gather(1, acciones)
        # 2 - Calculamos los objetivos (target) usando una red objetivo
        # No nos interesa calcular gradientes en esta red
        with torch.no_grad():
            # Q-values de los próximos estados (usando red objetivo)
            q_valores_siguientes = self.target_model(siguientes_estados)
            # Tomamos los máximos Q-values de los siguientes estados en el batch
            max_q_valor_siguientes = q_valores_siguientes.max(1, keepdim=True)[0]
            # Si el episodio terminó, no hay futuro (se anula el término γ * max Q)
            objetivos = recompensas + (1 - terminados) * self.GAMMA * max_q_valor_siguientes
        # 3 - Calculamos la pérdida entre el Q-valores actuales y los objetivos
        perdida = self.criterion(q_valor_accion, objetivos)
        # 4 - Calculamos los gradientes mediante backpropagation
        perdida.backward()
        # 5 - Actualizamos los parámetros de la red usando el optimizador
        self.optimizer.step()


Código para entrenar y evaluar el agente de DQN 
implementado.


In [8]:
# Ejecución del Agente DQN NO Determinista
env = gym.make('FrozenLake-v1',map_name="8x8", is_slippery=True)
agente_dqn = AgenteDQN()
entrenar_agente(agente_dqn, env, 10000)
resultados = evaluar_agente(agente_dqn, env, num_episodios=1000)
imprimir_resultados_evaluacion(resultados, "Agente DQN")
env.close()

Episodio 1000: Recompensa Promedio = 0.135, Longitud Promedio = 64.0
Episodio 2000: Recompensa Promedio = 0.379, Longitud Promedio = 60.3
Episodio 3000: Recompensa Promedio = 0.383, Longitud Promedio = 60.4
Episodio 4000: Recompensa Promedio = 0.399, Longitud Promedio = 61.3
Episodio 5000: Recompensa Promedio = 0.417, Longitud Promedio = 61.7
Episodio 6000: Recompensa Promedio = 0.402, Longitud Promedio = 61.9
Episodio 7000: Recompensa Promedio = 0.402, Longitud Promedio = 60.4
Episodio 8000: Recompensa Promedio = 0.399, Longitud Promedio = 62.5
Episodio 9000: Recompensa Promedio = 0.372, Longitud Promedio = 64.2
Episodio 10000: Recompensa Promedio = 0.407, Longitud Promedio = 61.3

Agente DQN - Resultados de Evaluación:
Tasa de Victorias: 48.3%
Recompensa Promedio: 0.483
Desviación Estándar: 0.500
Total de Victorias: 483


In [ ]:
# VALORES OPTIMOS ENCONTRADOS (A MANO, 1000 EPISODIOS DE ENTRENAMIENTO)
# TAMANO_ENTRADA = 16
# TAMANO_OCULTO = 128
# TAMANO_SALIDA = 4
# MEMORIA_MAX = 10000
# BATCH_SIZE = 64
# CONTADOR = 1000
# LR = 0.00025
# EPSILON_INICIAL = 1.0
# EPSILON_FINAL = 0.03
# EPSILON_DECAY = 0.999
# GAMMA = 0.95

In [ ]:
## Version que implementa memoria de experiencia y replay buffer
## Para 4x4 y 8x8

class DQN(nn.Module):
    """Clase auxiliar que implementa una Red Q Profunda con una capa oculta."""
    
    def __init__(self, tamano_entrada, tamano_oculto, tamano_salida):
        super(DQN, self).__init__()
        # Conexiones capa de entrada a oculta
        self.fc1 = nn.Linear(tamano_entrada, tamano_oculto)
        # Conexiones capa oculta a salida
        self.fc2 = nn.Linear(tamano_oculto, tamano_salida)
    
    def forward(self, x):
        # Se aplica ReLU a la combinacion lineal que llega a la capa oculta
        x = F.relu(self.fc1(x))
        # Se retorna la combinacion lineal que llega a la capa de salida
        return self.fc2(x)

class AgenteDQN(Agente):
    """Agente de Red Q Profunda."""

    def __init__(self, n):
        """Inicializa el agente DQN para un entorno FrozenLake de tamaño n x n."""
        # Parametros de la red
        self.tamano_entrada = n*n
        self.tamano_salida = 4
        if n == 4:
            self.tamano_oculto = 128
            self.memoria_max = 10000
            self.batch_size = 64
            self.contador_tope = 1000
            self.lr = 0.00025
            self.epsilon_final = 0.01
            self.epsilon_decay = 0.85
            self.gamma = 0.99
        elif n == 8:
            self.tamano_oculto = 128
            self.memoria_max = 20000
            self.batch_size = 128
            self.contador_tope = 1000
            self.lr = 0.0002
            self.epsilon_final = 0.01
            self.epsilon_decay = 0.90
            self.gamma = 0.95
        self.epsilon = 1.0
        self.counter = self.contador_tope
        # Inicialización de la red
        self.model = DQN(self.tamano_entrada, self.tamano_oculto, self.tamano_salida)
        self.target_model = DQN(self.tamano_entrada, self.tamano_oculto, self.tamano_salida)
        self.target_model.load_state_dict(self.model.state_dict())
        self.target_model.eval()
        self.criterion = nn.SmoothL1Loss()
        self.optimizer = optim.Adam(self.model.parameters(), lr=self.lr)
        self.memoria = deque(maxlen=self.memoria_max)

    def actualizar_red_objetivo(self):
        """Cada <self.contador> pasos actualiza los pesos de la red objetivo con los pesos de la red principal."""
        self.counter -= 1
        if self.counter == 0:
            self.target_model.load_state_dict(self.model.state_dict())
            self.counter = self.contador_tope

    def guardar_experiencia(self, estado, accion, recompensa, siguiente_estado, terminado):
        self.memoria.append((estado, accion, recompensa, siguiente_estado, terminado))
    
    def elegir_accion(self, estado):
        """Elige acción usando política epsilon-greedy."""
        # Tomamos una acción aleatoria con probabilidad epsilon (exploración)
        if random.random() < self.epsilon:
            accion = np.random.randint(self.tamano_salida)
        # Tomamos la acción con mayor Q-valor con probabilidad 1 - epsilon (explotación)
        else:
            tensor_estado = torch.eye(self.tamano_entrada)[estado].float().unsqueeze(0)
            with torch.no_grad():
                q_valores = self.model(tensor_estado)
                accion = torch.argmax(q_valores).item()
        return accion
    
    def aprender(self, estado, accion, recompensa, siguiente_estado, terminado):
        """Actualiza la red neuronal usando la experiencia (s, a, r, s')."""
        # Me aseguro que este el grad_enabled (Por alguna razón falla si se cambia de 4x4 a 8x8 sin esta linea)
        torch.set_grad_enabled(True)

        # Si termino el episodio actualizamos epsilon
        if terminado:
            self.epsilon = max(self.epsilon_final, self.epsilon * self.epsilon_decay)

        # Empezamos guardando la experiencia en la memoria
        self.guardar_experiencia(estado, accion, recompensa, siguiente_estado, terminado)
        self.actualizar_red_objetivo()
        if len(self.memoria) < self.batch_size:
            return
        
        # 0 - Limpiamos los gradientes acumulados antes de la actualización
        self.optimizer.zero_grad()
        # 1 - Tomamos un mini-batch aleatorio de la memoria y generamos tensores
        batch = random.sample(self.memoria, self.batch_size)
        estados, acciones, recompensas, siguientes_estados, terminados = zip(*batch)
        estados = torch.FloatTensor(np.identity(self.tamano_entrada)[list(estados)])
        acciones = torch.LongTensor(acciones).unsqueeze(1)
        recompensas = torch.FloatTensor(recompensas).unsqueeze(1)
        siguientes_estados = torch.FloatTensor(np.identity(self.tamano_entrada)[list(siguientes_estados)])
        terminados = torch.FloatTensor(terminados).unsqueeze(1)
        # 2 - Calculamos los Q-valores para estados actuales
        q_valores = self.model(estados)
        q_valor_accion = q_valores.gather(1, acciones)
        # 2 - Calculamos los objetivos (target) usando una red objetivo
        with torch.no_grad():
            q_valores_siguientes = self.target_model(siguientes_estados)
            max_q_valor_siguientes = q_valores_siguientes.max(1, keepdim=True)[0]
            objetivos = recompensas + (1 - terminados) * self.gamma * max_q_valor_siguientes
        # 3 - Calculamos la pérdida entre el Q-valores actuales y los objetivos
        perdida = self.criterion(q_valor_accion, objetivos)
        # 4 - Calculamos los gradientes mediante backpropagation
        perdida.backward()
        # 5 - Actualizamos los parámetros de la red usando el optimizador
        self.optimizer.step()

    def mostrar_hiperparametros(self):
        """Muestra los valores actuales de los hiperparámetros del agente."""
        print("Hiperparámetros del Agente DQN")
        print(f"tamaño_oculto   : {self.tamano_oculto}")
        print(f"memoria_max     : {self.memoria_max}")
        print(f"batch_size      : {self.batch_size}")
        print(f"contador_tope   : {self.contador_tope}")
        print(f"learning_rate   : {self.lr}")
        print(f"epsilon_final   : {self.epsilon_final}")
        print(f"epsilon_decay   : {self.epsilon_decay}")
        print(f"gamma           : {self.gamma}")


In [64]:
# Ejecución del Agente DQN NO Determinista
env = gym.make('FrozenLake-v1',map_name="4x4", is_slippery=True)
agente_dqn = AgenteDQN(4)
recompensas_episodios, longitudes_episodios = entrenar_agente(agente_dqn, env, 5000)
resultados = evaluar_agente(agente_dqn, env, num_episodios=5000)
imprimir_resultados_evaluacion(resultados, "Agente DQN")
env.close()

Episodio 500: Recompensa Promedio = 0.286, Longitud Promedio = 34.7
Episodio 1000: Recompensa Promedio = 0.366, Longitud Promedio = 33.1
Episodio 1500: Recompensa Promedio = 0.636, Longitud Promedio = 39.7
Episodio 2000: Recompensa Promedio = 0.638, Longitud Promedio = 40.0
Episodio 2500: Recompensa Promedio = 0.652, Longitud Promedio = 43.9
Episodio 3000: Recompensa Promedio = 0.666, Longitud Promedio = 41.8
Episodio 3500: Recompensa Promedio = 0.624, Longitud Promedio = 43.3
Episodio 4000: Recompensa Promedio = 0.658, Longitud Promedio = 42.7
Episodio 4500: Recompensa Promedio = 0.670, Longitud Promedio = 41.4
Episodio 5000: Recompensa Promedio = 0.632, Longitud Promedio = 42.8

Agente DQN - Resultados de Evaluación:
Tasa de Victorias: 61.1%
Recompensa Promedio: 0.611
Desviación Estándar: 0.487
Total de Victorias: 3056


In [66]:
resultados = evaluar_agente(agente_dqn, env, num_episodios=5000)
imprimir_resultados_evaluacion(resultados, "Agente DQN")


Agente DQN - Resultados de Evaluación:
Tasa de Victorias: 61.2%
Recompensa Promedio: 0.612
Desviación Estándar: 0.487
Total de Victorias: 3062


In [ ]:
# ESTO CAPAZ SE BORRA PORQUE NO FUE VARIABLE TERMINO SIEMPRE EN 20000 EPISODIOS

# Función de Entrenamiento de Agentes
def entrenar_agente_num_episodios_variable(agente, env, max_episodios=20000, intervalo=500, max_pasos=100, verbose=True):
    """
    Entrena un agente en el entorno.
    
    Args:
        agente: El agente a entrenar
        env: El entorno
        num_episodios: Número de episodios de entrenamiento
        max_pasos: Máximo de pasos por episodio
        verbose: Si imprimir el progreso
    
    Returns:
        list: Recompensas de episodios
    """
    recompensas_episodios = []
    recompensa_promedio = 0
    recompensa_promedio_ultimo_intervalo = -1
    longitudes_episodios = []

    sin_mejora = 0
    tolerancia = 5
    
    for episodio in range(max_episodios):
        estado, _ = env.reset()
        recompensa_total = 0
        pasos = 0
        
        for paso in range(max_pasos):
            accion = agente.elegir_accion(estado)
            siguiente_estado, recompensa, terminado, truncado, _ = env.step(accion)
            
            agente.aprender(estado, accion, recompensa, siguiente_estado, terminado or truncado)
            
            estado = siguiente_estado
            recompensa_total += recompensa
            pasos += 1
            
            if terminado or truncado:
                break
        
        recompensas_episodios.append(recompensa_total)
        longitudes_episodios.append(pasos)

        if episodio % intervalo == 0:
            recompensa_promedio = np.mean(recompensas_episodios[-intervalo:])
            if abs(recompensa_promedio - recompensa_promedio_ultimo_intervalo) < 0.01:
                sin_mejora += 1
            else:
                sin_mejora = 0

            if sin_mejora >= tolerancia:
                print(f"Entrenamiento detenido en {episodio} episodios.")
                break
            recompensa_promedio_ultimo_intervalo = recompensa_promedio
        
        if verbose and (episodio + 1) % intervalo == 0:
            longitud_promedio = np.mean(longitudes_episodios[-intervalo:])
            print(f"Entenando: Episodio {episodio + 1}: Recompensa Promedio = {recompensa_promedio:.3f}, Longitud Promedio = {longitud_promedio:.1f}, Sin mejora = {sin_mejora}", end="\r", flush=True)
        
    
    print(f"Entrenamiento detenido en {episodio} episodios.")
    return recompensas_episodios, longitudes_episodios

In [55]:
from itertools import product

def optimizar_hiperparametros(agente: Agente, param_ranges, env, num_episodios=1000, eval_episodios=1000):
    mejores_resultados = None
    mejor_agente = None
    param_grid = []

    # Generar combinaciones de hiperparámetros
    keys = param_ranges.keys()
    values = ( [round(float(v), 5) for v in param_ranges[k]] for k in keys )
    for combination in product(*values):
        params = dict(zip(keys, combination))
        param_grid.append(params)

    for params in param_grid:
        print(f"Probando configuración: {params}", flush=True)
        if isinstance(agente, AgenteDQN):
            n = int(agente.tamano_entrada**0.5)
            # Reiniciar agente DQN con tamaño adecuado
            agente = AgenteDQN(n)
            # Método get permite usar valores por defecto si no se especifican en params
            agente.tamano_oculto = params.get("tamano_oculto", agente.tamano_oculto)
            agente.memoria_max = params.get("memoria_max", agente.memoria_max)
            agente.batch_size = params.get("batch_size", agente.batch_size)
            agente.contador_tope = params.get("contador_tope", agente.contador_tope)
            agente.lr = params.get("lr", agente.lr)
            agente.epsilon_final = params.get("epsilon_final", agente.epsilon_final)
            agente.epsilon_decay = params.get("epsilon_decay", agente.epsilon_decay)
            agente.gamma = params.get("gamma", agente.gamma)
        # Evaluamos el agente con la configuración actual
        recompensas, _ = entrenar_agente_num_episodios_variable(agente, env)
        resultados = evaluar_agente(agente, env, num_episodios=eval_episodios)
        # Guardamos los mejores resultados
        if (mejores_resultados is None) or (resultados['recompensa_promedio'] > mejores_resultados['recompensa_promedio']):
            mejores_resultados = resultados
            mejor_agente = agente
        print(f"Recompensa promedio: {resultados['recompensa_promedio']}")

    print("\nMejor configuración encontrada:")
    mejor_agente.mostrar_hiperparametros()
    imprimir_resultados_evaluacion(mejores_resultados, "Optimización")
    return mejor_agente, mejores_resultados


In [52]:
param_ranges = {
    "tamano_oculto": [128, 256],
    "gamma": [0.90, 0.95, 0.99],
    "epsilon_decay" : [0.85, 0.90, 0.95],
    "lr": [0.0001, 0.00025, 0.0005]
}

In [58]:
# Ejecución del Agente DQN NO Determinista
env = gym.make('FrozenLake-v1',map_name="4x4", is_slippery=True)
agente_dqn = AgenteDQN(4)
agente_dqn, _ = optimizar_hiperparametros(agente_dqn, param_ranges, env)
resultados = evaluar_agente(agente_dqn, env, num_episodios=5000)
imprimir_resultados_evaluacion(resultados, "Agente DQN - mejores hiperparámetros")

Probando configuración: {'tamano_oculto': 128.0, 'gamma': 0.9, 'epsilon_decay': 0.85, 'lr': 0.0001}
Entrenamiento detenido en 19999 episodios.edio = 0.568, Longitud Promedio = 33.9, Sin mejora = 0
Recompensa promedio: 0.681
Probando configuración: {'tamano_oculto': 128.0, 'gamma': 0.9, 'epsilon_decay': 0.85, 'lr': 0.00025}
Entrenamiento detenido en 19999 episodios.edio = 0.582, Longitud Promedio = 34.3, Sin mejora = 0
Recompensa promedio: 0.701
Probando configuración: {'tamano_oculto': 128.0, 'gamma': 0.9, 'epsilon_decay': 0.85, 'lr': 0.0005}
Entrenamiento detenido en 19999 episodios.edio = 0.570, Longitud Promedio = 33.7, Sin mejora = 1
Recompensa promedio: 0.668
Probando configuración: {'tamano_oculto': 128.0, 'gamma': 0.9, 'epsilon_decay': 0.9, 'lr': 0.0001}
Entrenamiento detenido en 19999 episodios.edio = 0.572, Longitud Promedio = 34.4, Sin mejora = 0
Recompensa promedio: 0.484
Probando configuración: {'tamano_oculto': 128.0, 'gamma': 0.9, 'epsilon_decay': 0.9, 'lr': 0.00025}
Entr